# DSC288R - Exploratory Data Analysis
## Multi-Agent Graph RAG System for Explainable Financial Decision Support

**Project Group # 10**

**Authors:** Harsh Arya, Gabrielle Despaigne, Camila Paik, Raghav Vasappanavara

**Analysis Date:** February 2026

---

This notebook contains comprehensive Exploratory Data Analysis for our financial decision support system. The analysis covers:
1. Data Quality & Completeness
2. Price Distributions
3. Returns Distribution & Target Classes
4. Correlation Analysis
5. News Impact on Trading
6. Temporal Trends
7. Outlier Analysis
8. Feature Engineering Justifications

## 1. Setup and Data Loading

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# Create output directory
output_dir = Path('eda_outputs')
output_dir.mkdir(exist_ok=True)

print("Libraries loaded successfully!")

In [ ]:
# Load the aligned dataset from pipeline Stage 3
data_path = Path('../data/processed/data_aligned.parquet')

if data_path.exists():
    df = pd.read_parquet(data_path)
    print(f"Loaded dataset: {df.shape[0]:,} rows, {df.shape[1]} columns")
else:
    print(f"Data file not found at {data_path}")
    print("Please run the data pipeline first: python scripts/run_pipeline.py")

In [ ]:
# Display dataset info
print("Dataset Schema:")
print("=" * 60)
for col in df.columns:
    print(f"{col:25} {df[col].dtype}")
print("=" * 60)
print(f"\nTotal Columns: {len(df.columns)}")

In [ ]:
# Display sample records
print("Sample Records:")
df.head(3)

## 2. Data Quality & Completeness Analysis

Analyzing missing values and data coverage across all features.

In [ ]:
# Calculate missing values
missing_counts = df.isnull().sum()
missing_pct = (missing_counts / len(df) * 100).round(2)

# Create missing values summary
missing_df = pd.DataFrame({
    'Missing Count': missing_counts,
    'Missing %': missing_pct
}).sort_values('Missing %', ascending=False)

print("Missing Values Analysis:")
print("=" * 50)
print(missing_df[missing_df['Missing Count'] > 0])
print("\nColumns with 0% missing: ", (missing_pct == 0).sum())

In [ ]:
# Dataset characteristics
num_tickers = df['ticker'].nunique()
date_range_start = df['date'].min()
date_range_end = df['date'].max()
date_span_days = (pd.to_datetime(date_range_end) - pd.to_datetime(date_range_start)).days
date_span_years = date_span_days / 365.25

quality_summary = {
    'total_records': len(df),
    'num_tickers': num_tickers,
    'date_range': {
        'start': str(date_range_start),
        'end': str(date_range_end),
        'span_days': date_span_days
    },
    'missing_values': missing_df[missing_df['Missing Count'] > 0].to_dict(),
    'avg_records_per_ticker': round(len(df) / num_tickers, 2)
}

print("\nDataset Characteristics:")
print("=" * 50)
print(f"Total Records:        {quality_summary['total_records']:,}")
print(f"Number of Tickers:    {num_tickers}")
print(f"Date Range:           {date_range_start} to {date_range_end}")
print(f"Time Span:            {date_span_years:.1f} years ({date_span_days:,} days)")
print(f"Avg Records/Ticker:   {quality_summary['avg_records_per_ticker']:,.0f}")

# Save quality summary
with open(output_dir / '01_quality_summary.json', 'w') as f:
    json.dump(quality_summary, f, indent=2, default=str)
print("\nSaved: 01_quality_summary.json")

### INSIGHT: Data Quality Assessment

| Finding | Value | Implication |
|---------|-------|-------------|
| **Price Data** | 0% missing | Complete OHLCV for all 262K records - critical for technical analysis |
| **News Text** | 98.5% missing | Expected - not every stock has daily news coverage (1.46% with news) |
| **S&P 500 Context** | 86.4% coverage | Strong market context for relative performance analysis |
| **Date Range** | 14.2 years | Oct 2009 - Dec 2023: Captures multiple market cycles including COVID-19 |

## 3. Price Distribution Analysis

In [ ]:
# Price statistics
price_cols = ['open', 'high', 'low', 'close']
price_stats = df[price_cols + ['volume']].describe().round(2)
print("Price and Volume Statistics:")
print("=" * 70)
print(price_stats)

In [ ]:
# Create univariate distribution plots
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Price distributions
for idx, col in enumerate(price_cols):
    ax = axes[idx // 2, idx % 2] if idx < 2 else axes[1, idx - 2]
    df[col].hist(bins=100, ax=ax, color='steelblue', edgecolor='white', alpha=0.7)
    ax.set_title(f'{col.upper()} Distribution', fontsize=12, fontweight='bold')
    ax.set_xlabel('Price ($)')
    ax.set_ylabel('Frequency')
    ax.axvline(df[col].median(), color='red', linestyle='--', label=f'Median: ${df[col].median():.2f}')
    ax.axvline(df[col].mean(), color='green', linestyle='--', label=f'Mean: ${df[col].mean():.2f}')
    ax.legend(fontsize=9)

# Volume distribution (log scale)
ax = axes[1, 0]
df['volume'].apply(lambda x: np.log10(x + 1)).hist(bins=50, ax=ax, color='purple', edgecolor='white', alpha=0.7)
ax.set_title('VOLUME Distribution (log10)', fontsize=12, fontweight='bold')
ax.set_xlabel('log10(Volume)')
ax.set_ylabel('Frequency')

# Target distribution
ax = axes[1, 2]
target_counts = df['target'].value_counts()
colors = {'hold': 'steelblue', 'sell': '#e53e3e', 'buy': '#38a169'}
bars = ax.bar(target_counts.index, target_counts.values, color=[colors.get(x, 'gray') for x in target_counts.index])
ax.set_title('TARGET Distribution', fontsize=12, fontweight='bold')
ax.set_xlabel('Target Class')
ax.set_ylabel('Count')
for bar, count in zip(bars, target_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1000, 
            f'{count:,}\n({count/len(df)*100:.1f}%)', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig(output_dir / '02_univariate_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 02_univariate_distributions.png")

### INSIGHT: Price Distribution Analysis

**What We Found:**
- Large gap between mean ($62) and median ($27) reveals **right-skewed distribution**
- Price range spans $0.01 to $7,250 across 100 stocks
- Volume varies by orders of magnitude (1 to 470M shares)

**Action Taken:**
- Applied **per-ticker MinMaxScaler** for prices (0-1 range)
- Applied **per-ticker StandardScaler** for volume (z-score)
- This enables cross-stock comparison in model training

**Why this matters:** Without normalization, high-priced stocks would dominate model training.

## 4. Returns Distribution & Target Classes

In [ ]:
# Returns statistics
returns_stats = df['next_day_return'].describe()
print("Next-Day Returns Statistics:")
print("=" * 50)
print(f"Mean:          {returns_stats['mean']*100:.4f}%")
print(f"Median:        {returns_stats['50%']*100:.4f}%")
print(f"Std Dev:       {returns_stats['std']*100:.2f}%")
print(f"Min:           {returns_stats['min']*100:.2f}%")
print(f"Max:           {returns_stats['max']*100:.2f}%")
print(f"25th Pct:      {returns_stats['25%']*100:.2f}%")
print(f"75th Pct:      {returns_stats['75%']*100:.2f}%")

In [ ]:
# Returns distribution with thresholds
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
ax = axes[0]
returns_pct = df['next_day_return'] * 100
returns_pct_clipped = returns_pct.clip(-15, 15)  # Clip for visualization
ax.hist(returns_pct_clipped, bins=100, color='steelblue', edgecolor='white', alpha=0.7)
ax.axvline(-2, color='#e53e3e', linestyle='--', linewidth=2, label='Sell Threshold (-2%)')
ax.axvline(2, color='#38a169', linestyle='--', linewidth=2, label='Buy Threshold (+2%)')
ax.axvspan(-15, -2, alpha=0.1, color='red', label='SELL Zone')
ax.axvspan(2, 15, alpha=0.1, color='green', label='BUY Zone')
ax.set_title('Next-Day Returns Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Return (%)')
ax.set_ylabel('Frequency')
ax.legend(loc='upper right')
ax.set_xlim(-15, 15)

# Box plot
ax = axes[1]
bp = ax.boxplot(returns_pct_clipped.dropna(), vert=True, patch_artist=True)
bp['boxes'][0].set_facecolor('steelblue')
bp['boxes'][0].set_alpha(0.7)
ax.axhline(-2, color='#e53e3e', linestyle='--', linewidth=2)
ax.axhline(2, color='#38a169', linestyle='--', linewidth=2)
ax.set_title('Returns Box Plot', fontsize=14, fontweight='bold')
ax.set_ylabel('Return (%)')
ax.set_ylim(-15, 15)

plt.tight_layout()
plt.savefig(output_dir / '03_returns_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 03_returns_distribution.png")

In [ ]:
# Target distribution
target_dist = df['target'].value_counts()
target_pct = df['target'].value_counts(normalize=True) * 100

print("\nTarget Variable Distribution:")
print("=" * 50)
print(f"{'Target':<10} {'Count':>15} {'Percentage':>15}")
print("-" * 50)
for target in ['hold', 'sell', 'buy']:
    if target in target_dist.index:
        print(f"{target:<10} {target_dist[target]:>15,} {target_pct[target]:>14.1f}%")
print("-" * 50)
print(f"{'Total':<10} {len(df):>15,} {'100.0':>14}%")

### INSIGHT: Returns & Target Analysis

**What We Analyzed:**
- Distribution of next-day returns across 262K observations
- Created 3-class targets using ±2% thresholds
- Identified extreme movements (>10% gains/losses)

**What We Found:**
- **High volatility:** 34.66% std dev requires volatility features
- **Balanced targets:** 70% Hold, 15% Sell, 15% Buy
- **4,444 extreme moves:** Critical events for model training

**Features Created:**
- `volatility_20`: 20-day rolling std deviation
- `return_norm`: Standardized returns (z-score)
- `momentum_5/20`: Short/medium-term momentum

## 5. News Coverage Analysis

In [ ]:
# News coverage
has_news = df['text'].notna()
news_coverage = has_news.mean() * 100
news_count = has_news.sum()

print("News Coverage Analysis:")
print("=" * 50)
print(f"Records with news:      {news_count:,} ({news_coverage:.2f}%)")
print(f"Records without news:   {len(df) - news_count:,} ({100 - news_coverage:.2f}%)")

if 'news_count' in df.columns:
    avg_articles = df[df['news_count'] > 0]['news_count'].mean()
    print(f"Avg articles (when available): {avg_articles:.2f}")

In [ ]:
# News coverage visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Pie chart
ax = axes[0]
sizes = [len(df) - news_count, news_count]
labels = [f'No News\n({100-news_coverage:.1f}%)', f'With News\n({news_coverage:.2f}%)']
colors = ['#a0aec0', '#38a169']
ax.pie(sizes, labels=labels, colors=colors, autopct='', startangle=90)
ax.set_title('News Availability', fontsize=14, fontweight='bold')

# News count distribution
ax = axes[1]
if 'news_count' in df.columns:
    news_counts = df[df['news_count'] > 0]['news_count']
    ax.hist(news_counts, bins=20, color='#38a169', edgecolor='white', alpha=0.7)
    ax.set_title('Articles per Stock-Day (when available)', fontsize=14, fontweight='bold')
    ax.set_xlabel('Number of Articles')
    ax.set_ylabel('Frequency')
    ax.axvline(news_counts.mean(), color='red', linestyle='--', label=f'Mean: {news_counts.mean():.1f}')
    ax.legend()
else:
    ax.text(0.5, 0.5, 'News count data\nnot available', ha='center', va='center', fontsize=14)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig(output_dir / '04_news_coverage.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 04_news_coverage.png")

## 6. Correlation Analysis

In [ ]:
# Select numeric columns for correlation
numeric_cols = ['open', 'high', 'low', 'close', 'volume', 'sp500_return', 'next_day_return']
available_cols = [col for col in numeric_cols if col in df.columns]

# Calculate correlations
corr_matrix = df[available_cols].corr()

# Correlations with next_day_return
if 'next_day_return' in df.columns:
    target_corr = corr_matrix['next_day_return'].sort_values(ascending=False)
    print("Correlations with Next-Day Return:")
    print("=" * 50)
    for col, corr in target_corr.items():
        if col != 'next_day_return':
            print(f"{col:<20} {corr:>+.4f}")

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(10, 8))

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.3f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Correlation Coefficient'})
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(output_dir / '05_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 05_correlation_matrix.png")

### INSIGHT: Correlation Analysis Results

**Key Discovery:**
- **S&P 500 Return (+0.023)** is the strongest predictor of individual stock movements
- This confirms that market context is critical for stock-level predictions

**What Doesn't Work:**
- Raw price levels have negligible correlation (-0.003)
- Raw volume is not predictive (-0.001)
- Absolute values don't capture patterns

**Features Created Based on This:**
- `sp500_return`: Market return
- `excess_return`: Stock - Market
- `price_to_sma5/20`: Relative pricing
- `volume_ratio`: Relative volume

**Design Decision:** We use relative features (ratios, deviations from averages) instead of absolute values.

## 7. Price-Volume Relationship

In [ ]:
# Price-Volume scatter plot
fig, ax = plt.subplots(figsize=(10, 8))

# Sample for visualization
sample = df.sample(n=min(10000, len(df)), random_state=42)

scatter = ax.scatter(sample['close'], sample['volume'], 
                     alpha=0.3, c='steelblue', s=10)
ax.set_xlabel('Close Price ($)', fontsize=12)
ax.set_ylabel('Volume', fontsize=12)
ax.set_title('Price vs Volume Relationship (10K sample)', fontsize=14, fontweight='bold')
ax.set_yscale('log')

# Add trend line
z = np.polyfit(sample['close'], np.log10(sample['volume'] + 1), 1)
p = np.poly1d(z)

plt.tight_layout()
plt.savefig(output_dir / '06_price_volume_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 06_price_volume_scatter.png")

## 8. News Impact on Target Distribution

In [ ]:
# News impact analysis
df['has_news'] = df['text'].notna()

# Target distribution by news availability
target_by_news = df.groupby(['has_news', 'target']).size().unstack(fill_value=0)
target_by_news_pct = target_by_news.div(target_by_news.sum(axis=1), axis=0) * 100

print("Target Distribution by News Availability:")
print("=" * 60)
print(f"\n{'':15} {'Buy':>10} {'Hold':>10} {'Sell':>10}")
print("-" * 60)
for idx in [False, True]:
    label = 'With News' if idx else 'No News'
    row = target_by_news_pct.loc[idx]
    print(f"{label:15} {row.get('buy', 0):>9.1f}% {row.get('hold', 0):>9.1f}% {row.get('sell', 0):>9.1f}%")

In [ ]:
# Target relationships visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# By news availability
ax = axes[0]
x = np.arange(3)
width = 0.35
targets = ['buy', 'hold', 'sell']

no_news_pct = [target_by_news_pct.loc[False].get(t, 0) for t in targets]
with_news_pct = [target_by_news_pct.loc[True].get(t, 0) for t in targets]

bars1 = ax.bar(x - width/2, no_news_pct, width, label='No News', color='#a0aec0')
bars2 = ax.bar(x + width/2, with_news_pct, width, label='With News', color='#3182ce')

ax.set_ylabel('Percentage (%)')
ax.set_title('Target Distribution: News vs No News', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(['Buy', 'Hold', 'Sell'])
ax.legend()
ax.set_ylim(0, 80)

# By market direction (if sp500_return available)
ax = axes[1]
if 'sp500_return' in df.columns:
    df['market_direction'] = pd.cut(df['sp500_return'], 
                                     bins=[-np.inf, -0.001, 0.001, np.inf],
                                     labels=['Down', 'Flat', 'Up'])
    
    target_by_market = df.groupby(['market_direction', 'target']).size().unstack(fill_value=0)
    target_by_market_pct = target_by_market.div(target_by_market.sum(axis=1), axis=0) * 100
    
    target_by_market_pct[targets].plot(kind='bar', ax=ax, color=['#38a169', '#3182ce', '#e53e3e'])
    ax.set_title('Target by S&P 500 Direction', fontsize=14, fontweight='bold')
    ax.set_xlabel('Market Direction')
    ax.set_ylabel('Percentage (%)')
    ax.set_xticklabels(['Down', 'Flat', 'Up'], rotation=0)
    ax.legend(title='Target')
else:
    ax.text(0.5, 0.5, 'S&P 500 data\nnot available', ha='center', va='center', fontsize=14)

plt.tight_layout()
plt.savefig(output_dir / '07_target_relationships.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 07_target_relationships.png")

### INSIGHT: News Drives Volatility

**Key Finding:** Days with news show **10% more extreme movements** (buy/sell signals) compared to days without news.

| Target | No News | With News | Difference |
|--------|---------|-----------|------------|
| Buy | 14.8% | 18.6% | **+3.8 pp** |
| Hold | 69.9% | 60.9% | **-9.0 pp** |
| Sell | 15.3% | 20.5% | **+5.2 pp** |

**This validates the importance of news sentiment in our Graph RAG system.**

## 9. Temporal Analysis

In [ ]:
# Yearly statistics
df['year'] = pd.to_datetime(df['date']).dt.year

yearly_stats = df.groupby('year').agg({
    'close': ['count', 'mean'],
    'volume': 'mean',
    'next_day_return': 'mean'
}).round(4)

yearly_stats.columns = ['Records', 'Avg Price', 'Avg Volume', 'Avg Return']
yearly_stats['Avg Return'] = yearly_stats['Avg Return'] * 100

print("Yearly Statistics:")
print("=" * 70)
print(yearly_stats.to_string())

In [ ]:
# Temporal trends visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Records per year
ax = axes[0, 0]
yearly_stats['Records'].plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Records per Year', fontsize=12, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=45)

# Average price per year
ax = axes[0, 1]
yearly_stats['Avg Price'].plot(kind='line', ax=ax, marker='o', color='steelblue', linewidth=2)
ax.set_title('Average Price per Year', fontsize=12, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Price ($)')
ax.grid(True, alpha=0.3)

# Average volume per year
ax = axes[1, 0]
yearly_stats['Avg Volume'].plot(kind='bar', ax=ax, color='purple', edgecolor='white', alpha=0.7)
ax.set_title('Average Volume per Year', fontsize=12, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Volume')
ax.tick_params(axis='x', rotation=45)

# Highlight 2020
if 2020 in yearly_stats.index:
    idx_2020 = list(yearly_stats.index).index(2020)
    ax.patches[idx_2020].set_facecolor('#e53e3e')

# Average return per year
ax = axes[1, 1]
colors = ['#38a169' if r > 0 else '#e53e3e' for r in yearly_stats['Avg Return']]
yearly_stats['Avg Return'].plot(kind='bar', ax=ax, color=colors, edgecolor='white')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_title('Average Daily Return per Year (%)', fontsize=12, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Return (%)')
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(output_dir / '08_temporal_trends.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 08_temporal_trends.png")

### INSIGHT: Temporal Pattern Analysis

**COVID-19 Impact (2020):**
- **+1.35%** avg daily return (vs 0.2% typical)
- **3.5M** avg volume (2.4x normal)
- Price drop from $59 → $50 average

**Market Regime Changes:**
- 2014-2019: Stable growth period
- 2020: Extreme volatility (COVID crash/recovery)
- 2021-2023: Elevated volume persists

**Features to Capture This:**
- `volatility_20`: Detects regime changes
- `momentum_5/20`: Captures trend shifts
- `sma_5/20/50`: Multi-timeframe averages

## 10. Outlier Analysis

In [ ]:
# Price outliers using IQR method
Q1 = df['close'].quantile(0.25)
Q3 = df['close'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

price_outliers = df[(df['close'] < lower_bound) | (df['close'] > upper_bound)]

print("Price Outlier Detection (IQR Method):")
print("=" * 50)
print(f"Q1 (25th percentile):  ${Q1:.2f}")
print(f"Q3 (75th percentile):  ${Q3:.2f}")
print(f"IQR:                   ${IQR:.2f}")
print(f"Lower bound:           ${max(0, lower_bound):.2f}")
print(f"Upper bound:           ${upper_bound:.2f}")
print(f"Outliers detected:     {len(price_outliers):,} ({len(price_outliers)/len(df)*100:.2f}%)")

In [ ]:
# Extreme returns analysis
extreme_gains = df[df['next_day_return'] > 0.10]
extreme_losses = df[df['next_day_return'] < -0.10]
data_errors = df[abs(df['next_day_return']) > 0.50]  # >50% likely errors

print("\nExtreme Returns Analysis:")
print("=" * 50)
print(f"{'Category':<25} {'Count':>10} {'Percentage':>12}")
print("-" * 50)
print(f"{'Extreme Gains (>10%)':<25} {len(extreme_gains):>10,} {len(extreme_gains)/len(df)*100:>11.2f}%")
print(f"{'Extreme Losses (>10%)':<25} {len(extreme_losses):>10,} {len(extreme_losses)/len(df)*100:>11.2f}%")
print(f"{'Data Errors (>50%)':<25} {len(data_errors):>10,} {len(data_errors)/len(df)*100:>11.2f}%")
print("-" * 50)
print(f"{'Total Extreme Moves':<25} {len(extreme_gains) + len(extreme_losses):>10,}")

### INSIGHT: Outlier Detection & Treatment

**What We Analyzed:**
Used IQR method (1.5x threshold) to detect statistical outliers in price and return distributions.

**Retained (Valid Signals):**
- 25,372 price outliers (high-priced stocks)
- 4,444 extreme returns (real market events)
- These are valuable training examples!

**Removed (Data Errors):**
- 951 records with >50% daily change
- Likely stock splits or data quality issues
- Only 0.2% of data removed

**Design Decision:** Retain extreme but valid market moves while removing likely data errors.

## 11. EDA Summary: Key Findings → Feature Engineering Decisions

In [ ]:
# Create EDA insights summary
eda_insights = {
    'data_quality': {
        'total_records': len(df),
        'num_tickers': df['ticker'].nunique(),
        'date_range_years': date_span_years,
        'missing_data': '4 columns with missing values'
    },
    'distributions': {
        'target_balance': df['target'].value_counts(normalize=True).to_dict(),
        'returns_mean': float(df['next_day_return'].mean()),
        'returns_std': float(df['next_day_return'].std())
    },
    'anomalies': {
        'price_outliers': f"{len(price_outliers):,} ({len(price_outliers)/len(df)*100:.2f}%)",
        'extreme_gains': len(extreme_gains),
        'extreme_losses': len(extreme_losses)
    },
    'relationships': {
        'news_coverage': f"{news_coverage:.2f}%",
        'market_context_available': f"{(1 - df['sp500_return'].isna().mean())*100:.2f}%" if 'sp500_return' in df.columns else 'N/A'
    }
}

# Save insights
with open(output_dir / '09_eda_insights.json', 'w') as f:
    json.dump(eda_insights, f, indent=2, default=str)
print("Saved: 09_eda_insights.json")

In [ ]:
# Summary table
print("\n" + "=" * 80)
print("EDA SUMMARY: KEY FINDINGS → FEATURE ENGINEERING DECISIONS")
print("=" * 80)

findings = [
    ("S&P 500 correlation +0.023 (strongest)", "sp500_return, excess_return, market_up/down"),
    ("Right-skewed prices ($0.01 to $7,250)", "Per-ticker MinMaxScaler normalization"),
    ("High volatility (34.66% daily std)", "volatility_20, return_norm, momentum features"),
    ("10% more extreme moves on news days", "news_count, sentiment features for Graph RAG"),
    ("Raw prices have no predictive power", "price_to_sma5, price_to_sma20 ratios"),
    ("COVID-2020 shows regime change", "sma_5/20/50 multi-timeframe indicators")
]

print(f"\n{'Finding':<45} {'Feature(s) Created':<35}")
print("-" * 80)
for finding, feature in findings:
    print(f"{finding:<45} {feature:<35}")
print("-" * 80)
print("\nEach feature engineering decision is directly traced to a specific EDA finding.")

## 12. Files Generated

This EDA notebook generated the following artifacts:

| File | Description |
|------|-------------|
| `01_quality_summary.json` | Data quality metrics and missing value counts |
| `02_univariate_distributions.png` | 6 subplots: Price distributions (OHLC), Volume, Target labels |
| `03_returns_distribution.png` | Returns histogram with ±2% threshold lines and box plot |
| `04_news_coverage.png` | News availability analysis |
| `05_correlation_matrix.png` | Correlation heatmap for all numeric features |
| `06_price_volume_scatter.png` | Price-volume relationship scatter plot |
| `07_target_relationships.png` | Target distribution by news availability and market direction |
| `08_temporal_trends.png` | 4 subplots: Yearly trends in records, price, volume, returns |
| `09_eda_insights.json` | Aggregated statistical insights and key findings |

In [ ]:
# List all generated files
print("\nGenerated Files:")
print("=" * 50)
for f in sorted(output_dir.glob('*')):
    size = f.stat().st_size / 1024
    print(f"{f.name:<40} {size:>8.1f} KB")

---

**End of EDA Notebook**

All visualizations and insights from this notebook are incorporated into the DSC288R Progress Report.